In [190]:
# BPDRR code to generate the allocation of reparations per crew
# Mario Castro-Gama
# 2026-05-20
# 2026-06-15
# 2026-07-21

In [191]:
from pprint import pprint
import random

In [192]:
def allocate_crews(reparations, dmatrix, indexes, n_teams):
    """
    allocate_crews(reparations, dmatrix, indexes, n_teams):
    
    Description
       This is essentially a greedy load-balancing problem: 
       process jobs in the order given by indexes, and always assign the next job 
       to the crew with the smallest accumulated repair time.
    
    Input Parameters
    -----------------
    reparations : dict
        Dictionary {id: repair_time}
    dmatrix : dict
        Dictionary {id_i: {id_j: travel_time}}
    indexes : list
        List of ids indicating the allocation order, must have the same length of the keys of 'reparations'
    n_teams : int
        Number of crews, how many crews are you sending in the field to repair pipes

    Output / Returns
    ----------------
    dictionary with the pipe ids and the times of interventions for each crew. 
    dict { 'crew_1': { 'pipe_ids': [...], 'time_total': ...},
           'crew_2': { 'pipe_ids': [...], 'time_total': ...}, 
           ...
         }

    Developed by : Mario Castro-Gama, ir. MSc. PhD
    Last update  : 2026-05-20
                   2026-06-15, added travel time constant
                   2026-07-21, added travel tiem as function of 'dmatrix' distance matrix
    
    """

    # if each crew key is a string
    # crews = { f"crew_{i+1}": {'pipe_ids':  [], 
    #                           'time_k':    [], 
    #                           'time_t':    [], 
    #                           'time_0':    [],
    #                           'time_1':    [],
    #                           'time_total': 0,
    #                          } for i in range(n_teams)}

    # if each crew key is an int (starting at 1)
    crews = { i+1: {'pipe_ids':  [], 
                    'time_repair':    [], 
                    'time_travel':   [],
                    'time_0':    [],
                    'time_1':    [],
                    'time_total': 0,
                   } for i in range(n_teams)}

    nrep = len(reparations)

    print('')
    print('organize distances for each crew')
    new_controls = []
    for idx in indexes:
        if idx not in time_reparation:
            raise KeyError(f" Pipe ID '{idx}' not found in 'time_reparation' ")

        # Find the crew with the minimum accumulated time to allocate the next reparation
        # first available gets chosen
        crew_curr = min(crews, key = lambda c: crews[c]['time_total'])

        # current reparation time
        repair_time = reparations[idx]['t_r']

        # find the travel time between this pipe and the previous one
        if crews[crew_curr]['pipe_ids']==[]:
            travel_time = 0.5  # no previous pipe so give it 30 minutes
        else:
            # This is estimated from distance matrix 'dmatrix', that matrix is square and static for each damage scenario
            pipe_prev = crews[crew_curr]['pipe_ids'][-1]
            travel_time = dmatrix[pipe_prev][idx]
            print('Crew '+str(crew_curr)+', from '+pipe_prev+'-to-'+idx+' : '+str(travel_time))
        
        # Assign the reparation to each crew
        crews[crew_curr]['pipe_ids'].append(idx)
        crews[crew_curr]['time_repair'].append(repair_time)
        crews[crew_curr]['time_travel'].append(travel_time)
        crews[crew_curr]['time_0'].append(travel_time + crews[crew_curr]['time_total'])
        crews[crew_curr]['time_1'].append(repair_time + crews[crew_curr]['time_0'][-1])
        crews[crew_curr]['time_total'] += repair_time + travel_time

        new_controls.append(f"; Crew {crew_curr} - Pipe {idx}\n")
        new_controls.append(f"LINK {idx}_A CLOSED AT TIME {crews[crew_curr]['time_0'][-1]}\n") # close Pipe_id_<A> when arriving to the location
        new_controls.append(f"LINK {idx}_B CLOSED AT TIME {crews[crew_curr]['time_0'][-1]}\n") # close Pipe_id_<B> when arriving to the location
        new_controls.append(f"LINK {idx} OPEN AT TIME {crews[crew_curr]['time_1'][-1]}\n")     # Open Pipe_id only after time of reparation
        
    return crews, new_controls

In [193]:
# Example 1 - very simple data
# dictionary with pipe_ids (str) and the time of reparation (int) in hours
# time of reparation  = time_repair + time_travel_ij + time_shift

n_rep   = 6
n_crews = 2
time_reparation = {'0': {'t_r':5}, 
                   '1': {'t_r':8}, 
                   '2': {'t_r':3}, 
                   '3': {'t_r':7}, 
                   '4': {'t_r':4}, 
                   '5': {'t_r':6},
                  }

# create artificial values for distance matrix
dmatrix = {f'{i}': {f'{j}': 0.00 if i==j else 0.25*random.randint(1, 8) for j in range(n_rep)} for i in range(n_rep)}
print('')
print('distance matrix')
pprint(dmatrix)

# this is just a permutation of the list of pipes to repair
indexes = ['2','0','5','3','1','4']
print('')
print('indexes')
print(indexes)

# apply the function of greedy allocation
crews, new_controls = allocate_crews(time_reparation, dmatrix, indexes, n_teams=n_crews)

print('')
print('crews')
pprint(crews)

print('')
print('controls')
pprint(new_controls)


distance matrix
{'0': {'0': 0.0, '1': 1.0, '2': 0.5, '3': 1.0, '4': 1.0, '5': 0.75},
 '1': {'0': 0.5, '1': 0.0, '2': 0.5, '3': 1.75, '4': 0.5, '5': 0.75},
 '2': {'0': 1.25, '1': 1.0, '2': 0.0, '3': 0.25, '4': 1.5, '5': 0.75},
 '3': {'0': 1.25, '1': 2.0, '2': 0.75, '3': 0.0, '4': 1.5, '5': 0.25},
 '4': {'0': 1.0, '1': 2.0, '2': 0.25, '3': 1.75, '4': 0.0, '5': 0.25},
 '5': {'0': 2.0, '1': 0.75, '2': 1.75, '3': 0.25, '4': 0.5, '5': 0.0}}

indexes
['2', '0', '5', '3', '1', '4']

organize distances for each crew
Crew 1, from 2-to-5 : 0.75
Crew 2, from 0-to-3 : 1.0
Crew 1, from 5-to-1 : 0.75
Crew 2, from 3-to-4 : 1.5

crews
{1: {'pipe_ids': ['2', '5', '1'],
     'time_0': [0.5, 4.25, 11.0],
     'time_1': [3.5, 10.25, 19.0],
     'time_repair': [3, 6, 8],
     'time_total': 19.0,
     'time_travel': [0.5, 0.75, 0.75]},
 2: {'pipe_ids': ['0', '3', '4'],
     'time_0': [0.5, 6.5, 15.0],
     'time_1': [5.5, 13.5, 19.0],
     'time_repair': [5, 7, 4],
     'time_total': 19.0,
     'time_travel

In [194]:
# Example 2 - larger instance with a larger number of pipes to repair and random values of reparation time
# generate randomly the times but keep the indexes as ordered strings

n_crews = 50
n_reparations = 150
time_reparation = { str(i): {'t_r':0.25*random.randint(1, 24),
                             't_t':0.25*random.randint(1, 4),
                            } for i in range(n_reparations)} # times are in hours every 15 min
#pprint(time_reparation)

# create artificial values for distance matrix
dmatrix = {f'{i}': {f'{j}': 0.00 if i==j else 0.25*random.randint(1, 8) for j in range(n_reparations)} for i in range(n_reparations)}
#pprint(dmatrix)

# generate a random permutation, one would expect to get this directly from the optimization (PYMOO)
indexes = random.sample(list(time_reparation.keys()), n_reparations)
print('Show permutation of reparations')
print(indexes)
print(' ')

# apply the greedy allocation to the dataset
crews, new_controls = allocate_crews(time_reparation, dmatrix, indexes, n_teams = n_crews)
#print('')
#print('Show the allocation of reparations to crews')
#pprint(crews)
print('')
print('[CONTROLS]')
pprint(new_controls)

Show permutation of reparations
['124', '45', '20', '14', '62', '100', '123', '88', '71', '42', '48', '5', '98', '2', '69', '47', '46', '117', '35', '11', '73', '41', '121', '107', '106', '50', '89', '93', '28', '101', '9', '8', '136', '149', '81', '72', '146', '78', '58', '115', '22', '145', '55', '130', '140', '52', '37', '6', '131', '122', '125', '134', '82', '54', '51', '86', '114', '23', '56', '80', '38', '18', '24', '143', '108', '67', '74', '111', '103', '119', '29', '3', '60', '7', '59', '141', '96', '84', '12', '33', '133', '77', '16', '43', '94', '32', '15', '4', '91', '105', '68', '132', '57', '120', '25', '75', '110', '128', '92', '26', '129', '30', '90', '39', '66', '70', '138', '99', '102', '34', '27', '137', '127', '148', '95', '83', '87', '65', '76', '61', '13', '44', '147', '10', '0', '118', '112', '63', '97', '1', '116', '64', '31', '21', '53', '126', '135', '144', '109', '49', '142', '85', '36', '104', '19', '139', '79', '40', '17', '113']
 

organize distances for e

In [189]:
pprint(crews)

{1: {'pipe_ids': ['42', '24', '146'],
     'time_0': [0.5, 3.5, 8.0],
     'time_1': [3.25, 6.0, 11.75],
     'time_repair': [2.75, 2.5, 3.75],
     'time_total': 11.75,
     'time_travel': [0.5, 0.25, 2.0]},
 2: {'pipe_ids': ['13', '19'],
     'time_0': [0.5, 7.5],
     'time_1': [6.5, 13.25],
     'time_repair': [6.0, 5.75],
     'time_total': 13.25,
     'time_travel': [0.5, 1.0]},
 3: {'pipe_ids': ['104', '12', '113'],
     'time_0': [0.5, 6.0, 11.0],
     'time_1': [5.0, 10.25, 14.5],
     'time_repair': [4.5, 4.25, 3.5],
     'time_total': 14.5,
     'time_travel': [0.5, 1.0, 0.75]},
 4: {'pipe_ids': ['133', '28'],
     'time_0': [0.5, 5.75],
     'time_1': [4.25, 10.75],
     'time_repair': [3.75, 5.0],
     'time_total': 10.75,
     'time_travel': [0.5, 1.5]},
 5: {'pipe_ids': ['27', '129'],
     'time_0': [0.5, 6.25],
     'time_1': [5.0, 10.75],
     'time_repair': [4.5, 4.5],
     'time_total': 10.75,
     'time_travel': [0.5, 1.25]},
 6: {'pipe_ids': ['88', '132', '139'],
 